In [2]:
#*****************************************Import********************************************
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import warnings
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from sklearn import preprocessing
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from datetime import datetime

df=pd.read_csv("Droidware.csv")

ROWS = len(df.axes[0]) 
COLUMNS = len(df.axes[1])
df = df.replace([np.inf, -np.inf], np.nan)
df=df.dropna()
LABEL = df.iloc[:,-1:].columns[0]
numCols = df.select_dtypes(include=['float64','int64']).columns
df = pd.DataFrame(df[numCols]).copy()
print("Row-Column Count before cleaning: (", ROWS , " , ",  COLUMNS , ")")
df.iloc[:,-1:].value_counts()

Row-Column Count before cleaning: ( 253527  ,  70 )


Label
1        129946
0        123573
Name: count, dtype: int64

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix)
import pywt  # For wavelet transform

# Load dataset
y = df['Label'].values
X = df.drop(columns=['Label']).values

# Preprocessing
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

def frequency_transform(data):
    """ Apply FFT and Wavelet transform """
    transformed_data = []
    for sample in data:
        fft_features = np.abs(np.fft.fft(sample))[:len(sample)//2]
        wavelet_coeffs, _ = pywt.dwt(sample, 'haar')
        transformed_data.append(np.concatenate([fft_features, wavelet_coeffs]))
    return np.array(transformed_data)

X_train_ft = frequency_transform(X_train)
X_test_ft = frequency_transform(X_test)

# Convert to Torch tensors
X_train_torch = torch.tensor(X_train_ft, dtype=torch.float32)
X_test_torch = torch.tensor(X_test_ft, dtype=torch.float32)
y_train_torch = torch.tensor(y_train, dtype=torch.float32)
y_test_torch = torch.tensor(y_test, dtype=torch.float32)

# Define Generator
class Generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Tanh()
        )
    def forward(self, z):
        return self.model(z)

# Define Transformer-based Discriminator
class TransformerDiscriminator(nn.Module):
    def __init__(self, input_dim):
        super(TransformerDiscriminator, self).__init__()
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=4)
        self.transformer = nn.TransformerEncoder(self.encoder_layer, num_layers=3)
        self.fc = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        x = self.transformer(x.unsqueeze(1)).squeeze(1)
        return torch.sigmoid(self.fc(x))

# Hyperparameters
gen_input_dim = 100
gan_output_dim = X_train_ft.shape[1]
discriminator_input_dim = X_train_ft.shape[1]

generator = Generator(gen_input_dim, gan_output_dim)
discriminator = TransformerDiscriminator(discriminator_input_dim)

# Optimizers
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Loss function
criterion = nn.BCELoss()

# Training Loop
num_epochs = 200
for epoch in range(num_epochs):
    # Train Discriminator
    real_labels = torch.ones((X_train_torch.size(0), 1))
    fake_labels = torch.zeros((X_train_torch.size(0), 1))
    z = torch.randn((X_train_torch.size(0), gen_input_dim))
    generated_data = generator(z)
    
    d_optimizer.zero_grad()
    real_loss = criterion(discriminator(X_train_torch), real_labels)
    fake_loss = criterion(discriminator(generated_data.detach()), fake_labels)
    d_loss = real_loss + fake_loss
    d_loss.backward()
    d_optimizer.step()
    
    # Train Generator
    g_optimizer.zero_grad()
    g_loss = criterion(discriminator(generated_data), real_labels)
    g_loss.backward()
    g_optimizer.step()
    
    if epoch % 10 == 0:
        print(f'Epoch [{epoch}/{num_epochs}], D_Loss: {d_loss.item()}, G_Loss: {g_loss.item()}')

# Evaluate the Model
with torch.no_grad():
    y_pred_prob = discriminator(X_test_torch).numpy().flatten()
    y_pred = (y_pred_prob > 0.5).astype(int)

# Compute Performance Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
conf_matrix = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = conf_matrix.ravel()
markedness = precision + recall - 1
youden_j = recall + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * recall)

# Print Results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"MCC: {mcc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Markedness Measure: {markedness:.4f}")
print(f"Youden’s J Statistic: {youden_j:.4f}")
print(f"Fowlkes-Mallows Index: {fmi:.4f}")

RuntimeError: [enforce fail at alloc_cpu.cpp:114] data. DefaultCPUAllocator: not enough memory: you tried to allocate 658142787600 bytes.